<img src='https://data.actris.eu/static/img/actris-dc-logo.png' width=400 align=right>

# Search with ACTRIS Metadata Rest API 

The goal of this notebook is to provide a guide on how to access data through the ACTRIS Metadata Rest API. This is a machine to machine approch to accessing data and is suited when you plan to access large amounts of data or only want to use a programming interface to access data. 

Let's get started!

### Using ACTRIS metadata catalog REST API

ACTRIS metadata catalog REST API: https://prod-actris-md2.nilu.no/index.html

The ACTRIS Rest API uses the ACTRIS vocabulary for several of the search criteria, the vocabulary can be found here: https://vocabulary.actris.nilu.no/skosmos/actris_vocab/en/

### Import libraries

In [2]:
# Library for working with multi-dimensional arrays 
import pandas as pd

# Libraries for working with JSON files, making HTTP requests, and handling file system operations
import requests

# Library for creating progress bars
from tqdm.notebook import tqdm

## Accessing metadata

The full ACTRIS metadata catalog can be accessed with https://prod-actris-md2.nilu.no/metadata/. 

The API uses pagination, so to get the full catalogue of what you are searching for you have to go through all pages untill you hit an empty page.

### Accessing one page of metadata

In [3]:
# get first page of metadata from ACTRIS metadata archive
response = requests.get("https://prod-actris-md2.nilu.no/metadata/page/0") 
metadata_archive = response.json()  

In [4]:
metadata_archive[0] # show one metadata element

{'id': '67479700a34b92517e800417',
 'md_metadata': {'provider': {'name': 'ARES',
   'atom': 'http://localhost:5009/providers/10'},
  'file_identifier': 'EARLINET_AerRemSen_waw_Lev01_b0355_202211082315_202211082359_v01_qc03.nc',
  'language': 'en',
  'hierarchy_level': 'dataset',
  'online_resource': {'linkage': 'https://data.earlinet.org/'},
  'datestamp': '2023-11-13T12:07:00Z',
  'created': '2024-06-11T10:25:46Z',
  'contact': [{'first_name': 'Lucia',
    'last_name': 'Mona',
    'organisation_name': 'CNR-IMAA',
    'role_code': ['custodian',
     'distributor',
     'pointOfContact',
     'processor',
     'publisher',
     'resourceProvider'],
    'country_code': 'IT',
    'delivery_point': 'Contrada S.Loja, Zona Industriale - Tito Scalo I-85050 Potenza',
    'address_city': 'Potenza',
    'email': 'lucia.mona@cnr.it',
    'position_name': 'Senior Researcher'}]},
 'md_identification': {'abstract': 'Profiles of aerosol optical properties',
  'title': 'Aerosol particle backscatter pr

In [5]:
# Most of these keys consists of a new dictonary with metadata information. 
# An example is md_metadata 
md_list = []
for f in metadata_archive:
    md_list.append(f['md_metadata']) 
df_md_metadata = pd.DataFrame.from_records(md_list)

df_md_metadata.iloc[0] #only show first element in list of metadata

provider           {'name': 'ARES', 'atom': 'http://localhost:500...
file_identifier    EARLINET_AerRemSen_waw_Lev01_b0355_20221108231...
language                                                          en
hierarchy_level                                              dataset
online_resource            {'linkage': 'https://data.earlinet.org/'}
datestamp                                       2023-11-13T12:07:00Z
created                                         2024-06-11T10:25:46Z
contact            [{'first_name': 'Lucia', 'last_name': 'Mona', ...
Name: 0, dtype: object

In [6]:
# Above the column 'contact' includes more information about a contact person for each dataset. 

df_md_metadata.iloc[0]['contact'] # show contact information for first dataset

[{'first_name': 'Lucia',
  'last_name': 'Mona',
  'organisation_name': 'CNR-IMAA',
  'role_code': ['custodian',
   'distributor',
   'pointOfContact',
   'processor',
   'publisher',
   'resourceProvider'],
  'country_code': 'IT',
  'delivery_point': 'Contrada S.Loja, Zona Industriale - Tito Scalo I-85050 Potenza',
  'address_city': 'Potenza',
  'email': 'lucia.mona@cnr.it',
  'position_name': 'Senior Researcher'}]

In [7]:
# Another example of extracting metadata, here the content information.
files_list = []
for f in metadata_archive:
    url = f['md_content_information']
    files_list.append(url)
    
df_content_information = pd.DataFrame.from_records(files_list)
# Displays the content information for all datasets from the first page of the metadata archive
df_content_information 


,attribute_descriptions,content_type
0,[aerosol particle backscatter coefficient],physicalMeasurement
1,"[aerosol particle backscatter coefficient, aer...",physicalMeasurement
2,"[aerosol particle backscatter coefficient, aer...",physicalMeasurement
3,"[aerosol particle backscatter coefficient, aer...",physicalMeasurement
4,[aerosol particle light extinction coefficient],physicalMeasurement
5,"[aerosol particle backscatter coefficient, aer...",physicalMeasurement
6,[aerosol particle light extinction coefficient],physicalMeasurement
7,"[aerosol particle backscatter coefficient, aer...",physicalMeasurement
8,[aerosol particle backscatter coefficient],physicalMeasurement
9,[aerosol particle light extinction coefficient],physicalMeasurement


In [8]:
# Another example of extracting metadata, here the distribution information.
# The distribution information includes data format, dataset url, protocol, restrictions and more.

files_list = []
for f in metadata_archive:
    url = f['md_distribution_information'][0]
    files_list.append(url)
    
df_distribution_information = pd.DataFrame.from_records(files_list)
df_distribution_information.iloc[0] #show the distribution information for the first dataset. 
# If you wish to see distribution information about all metadata in your list, remove .iloc[0]

data_format                                                       netcdf
version_data_format                                       HDF5 (NETCDF4)
dataset_url            https://data.earlinet.org/api/services/restapi...
protocol                                                            HTTP
function                                                        download
restriction                                               {'set': False}
transfersize                                                   41.671875
description                                 Direct download of data file
Name: 0, dtype: object

### Accessing spesific metadata, and working with pagination

In [9]:
# Get all metadata from lidar instruments at Jungfraujoch, Switzerland
i = 0
metadata_archive = []
pbar = tqdm(desc="Fetching metadata pages", unit="page") # create a progress bar

while True:
    # request metadata from lidar, CH = Switzerland and facility Jungfraujoch
    response = requests.get(f"https://prod-actris-md2.nilu.no/metadata/instrument/lidar/country/CH/facility/mmee/page/{i}") 
    if not response.json():
        break
    metadata_archive += response.json()
    i += 1
    pbar.update(1)

pbar.close()

Fetching metadata pages: 0page [00:00, ?page/s]

In [10]:
metadata_archive[0]

{'id': '6747a1eca34b92517e81db8b',
 'md_metadata': {'provider': {'name': 'ARES',
   'atom': 'http://localhost:5009/providers/10'},
  'file_identifier': 'EARLINET_AerRemSen_jfj_Lev02_b0532_200205201800_200205201830_v02_qc03.nc',
  'language': 'en',
  'hierarchy_level': 'dataset',
  'online_resource': {'linkage': 'https://data.earlinet.org/'},
  'datestamp': '2021-06-15T10:03:00Z',
  'created': '2024-06-14T09:17:04Z',
  'contact': [{'first_name': 'Lucia',
    'last_name': 'Mona',
    'organisation_name': 'CNR-IMAA',
    'role_code': ['custodian',
     'distributor',
     'pointOfContact',
     'processor',
     'publisher',
     'resourceProvider'],
    'country_code': 'IT',
    'delivery_point': 'Contrada S.Loja, Zona Industriale - Tito Scalo I-85050 Potenza',
    'address_city': 'Potenza',
    'email': 'lucia.mona@cnr.it',
    'position_name': 'Senior Researcher'}]},
 'md_identification': {'abstract': 'Profiles of aerosol optical properties',
  'title': 'Aerosol particle backscatter pr

You can see all the different ways to specify metadata at https://prod-actris-md2.nilu.no/ 